# AI-Driven DTM Generator — End-to-End Notebook

## Pipeline

```
Raw LiDAR Point Cloud Tiles
        ↓
Feature Engineering  [ΔZ · roughness · slope · density]
        ↓
PointNet++ MSG  (Ground / Non-Ground Segmentation)
        ↓
Ground Points Extracted  (threshold-optimised)
        ↓
Rasterize → Fill Holes → Fill Pits → Gaussian Smooth
        ↓
✅  Clean DTM  (.npy per tile + inference script for web integration)
```

## P100 Setup Note  *(run Cell 1 ONCE, then Change Kernel)*
| Python | Problem | Fix (Cell 1 handles automatically) |
|---|---|---|
| 3.12 (Kaggle default) | No PyTorch wheel supports both py312 + P100 (sm_60) | Installs Python 3.10 venv + torch 2.0.1+cu117 as a Jupyter kernel |

After Cell 1 completes: **Kernel menu → Change Kernel → "DTM-P100 (py3.10)"**, then run from Cell 2.  
If you already use T4 or A100: skip Cell 1 entirely.

## What this notebook produces
| Output | Path | Use |
|---|---|---|
| `best_model.pth` | `/kaggle/working/logs/` | Load in PyTorch for inference |
| `inference_config.json` | `/kaggle/working/dtm_output/` | Threshold, arch params |
| `dtm_inference.py` | `/kaggle/working/dtm_output/` | Drop-in script for web backend |
| DTM tiles (`.npy`) | `/kaggle/working/dtm_output/val/` | Sample clean terrain models |
| `dtm_model_outputs.zip` | `/kaggle/working/` | Everything in one download |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1  P100 One-Time Fix  (run once, ~5 min, then CHANGE KERNEL)
# ─────────────────────────────────────────────────────────────────────────────
# WHY THIS IS NEEDED
#   Kaggle ships Python 3.12.  PyTorch 2.0.1 (last version supporting P100/sm_60)
#   was released before Python 3.12 existed — no cp312 wheels exist anywhere.
#   This cell installs Python 3.10 + PyTorch 2.0.1 as a separate Jupyter kernel.
#
# WHAT TO DO AFTER THIS CELL COMPLETES
#   Kernel menu  →  Change Kernel  →  "DTM-P100 (py3.10)"
#   Then run ALL cells from Cell 2 onward.  Do NOT use Restart — CHANGE KERNEL.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys

def _q(cmd, t=600):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=t)

ENV  = "/kaggle/working/dtm_env"
PY   = f"{ENV}/bin/python"
PIP  = f"{ENV}/bin/pip"
WHL  = ("https://download.pytorch.org/whl/cu117/"
        "torch-2.0.1%2Bcu117-cp310-cp310-linux_x86_64.whl")

# Check if already done
r = _q(f"{PY} -c \"import torch; x=torch.randn(4,4,device='cuda'); (x@x).sum().item(); print('ok')\"")
if "ok" in r.stdout:
    print("✅ Already installed and GPU verified.  Change Kernel → 'DTM-P100 (py3.10)'")
else:
    print("Setting up P100-compatible environment (one-time, ~5 min)...")
    steps = [
        ("Python 3.10 via apt",
         "add-apt-repository -y ppa:deadsnakes/ppa -qq 2>/dev/null; "
         "apt-get install -qqy python3.10 python3.10-venv python3.10-dev 2>/dev/null"),
        ("Create venv",
         f"python3.10 -m venv {ENV}"),
        ("Upgrade pip",
         f"{PIP} install -q --upgrade pip"),
        ("Install PyTorch 2.0.1+cu117 (2.4 GB)",
         f"{PIP} install -q {WHL}"),
        ("Install deps",
         f"{PIP} install -q ipykernel tqdm numpy matplotlib scipy psutil"),
        ("Register kernel",
         f"{PY} -m ipykernel install --user --name dtm_p100 "
         f"--display-name 'DTM-P100 (py3.10)'"),
    ]
    ok = True
    for name, cmd in steps:
        r = _q(cmd, t=700)
        status = "✅" if r.returncode == 0 else "❌"
        print(f"  {status} {name}")
        if r.returncode != 0:
            print(f"     {r.stderr.strip()[-250:]}")
            ok = False; break

    if ok:
        r2 = _q(f"{PY} -c \"import torch; x=torch.randn(4,4,device='cuda');"
                "(x@x).sum().item(); "
                "print(f'torch={{torch.__version__}} sm_{{torch.cuda.get_device_capability(0)}}')\"")
        print(f"\n  GPU test: {r2.stdout.strip() or r2.stderr.strip()[-120:]}")
        print("\n" + "="*60)
        print("  ✅ Done!  NOW:")
        print("  Kernel menu  →  Change Kernel  →  'DTM-P100 (py3.10)'")
        print("  Then run all cells from Cell 2 onward.")
        print("="*60)
    else:
        print("\n❌ Setup failed. Alternatives:")
        print("  1. Kaggle Settings → Accelerator → T4 GPU  (works with any PyTorch)")
        print("  2. Kaggle Settings → Environment → Legacy GPU  (Python 3.10 pre-installed)")


In [ ]:
# ─── Cell 2  GPU Verification + Memory Helpers ───────────────────────────────
import torch, gc, os, time
import numpy as np

print(f"Python  : {__import__('sys').version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not found.\n"
        "Did you select kernel 'DTM-P100 (py3.10)' after running Cell 1?\n"
        "Use: Kernel menu → Change Kernel → DTM-P100 (py3.10)"
    )

cap = torch.cuda.get_device_capability(0)
try:
    _x = torch.randn(64, 64, device="cuda")
    (_x @ _x).sum().item()
    torch.cuda.synchronize()
    del _x
    kernel_ok = True
except Exception as e:
    kernel_ok = False
    raise RuntimeError(
        f"GPU detected but CUDA kernels fail: {e}\n"
        "Wrong kernel selected. Use: Kernel menu → Change Kernel → DTM-P100 (py3.10)\n"
        "OR: Kaggle Settings → Accelerator → T4 GPU (works with any PyTorch)"
    )

print(f"GPU     : {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Kernels : ✅ executing")
if cap[0] < 7:
    print("AMP     : fp16  (sm_60 does not support TF32 — fp16 is fine)")

torch.backends.cudnn.benchmark = True
device      = torch.device("cuda")
DEVICE_NAME = "GPU"


def mem_report(tag=""):
    try:
        import psutil
        rss   = psutil.Process(os.getpid()).memory_info().rss / 1e9
        rtot  = psutil.virtual_memory().total / 1e9
    except Exception:
        rss = rtot = 0.0
    vu = torch.cuda.memory_allocated() / 1e9
    vt = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[MEM {tag:<20s}]  GPU {vu:.2f}/{vt:.1f} GB  RAM {rss:.2f}/{rtot:.1f} GB")


def free_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


mem_report("startup")


In [ ]:
# ─── Cell 3  Dataset Detection ───────────────────────────────────────────────
from pathlib import Path
from glob    import glob

# ── Set your Kaggle dataset path here ────────────────────────────────────────
DATASET_ROOT     = "/kaggle/input/datasets/jaideepchouhan/point-cloud-data-of-10-indian-villages/Training"
DATASET_ZIP_PATH = ""   # leave blank if using extracted dataset
# ─────────────────────────────────────────────────────────────────────────────

DATA_SOURCE_MODE = None
TRAINING_ROOT    = None
DATA_ZIP_PATH    = None
ZIP_ROOT_PREFIX  = ""

def _has_split(root):
    r = Path(root)
    return (r / "train").exists() and (r / "val").exists()

if DATASET_ROOT and Path(DATASET_ROOT).exists():
    if not _has_split(DATASET_ROOT):
        raise FileNotFoundError(f"No train/val under: {DATASET_ROOT}")
    DATA_SOURCE_MODE, TRAINING_ROOT = "dir", Path(DATASET_ROOT)

elif DATASET_ZIP_PATH:
    DATA_SOURCE_MODE, DATA_ZIP_PATH = "zip", Path(DATASET_ZIP_PATH)

else:
    import zipfile, collections
    dirs = [Path(p) for p in glob("/kaggle/input/**", recursive=True)
            if Path(p).is_dir() and _has_split(Path(p))]
    if len(dirs) == 1:
        DATA_SOURCE_MODE, TRAINING_ROOT = "dir", dirs[0]
    elif len(dirs) > 1:
        w = [d for d in dirs if (d / "split_manifest.json").exists()]
        if w: DATA_SOURCE_MODE, TRAINING_ROOT = "dir", w[0]
    if DATA_SOURCE_MODE is None:
        zips = sorted(glob("/kaggle/input/**/*.zip", recursive=True))
        if len(zips) == 1:
            DATA_SOURCE_MODE, DATA_ZIP_PATH = "zip", Path(zips[0])

if DATA_SOURCE_MODE == "dir":
    tr = list((TRAINING_ROOT / "train").glob("tile_*"))
    va = list((TRAINING_ROOT / "val").glob("tile_*"))
    print(f"✅ Dataset (dir) : {TRAINING_ROOT}")
    print(f"   Train tiles   : {len(tr)}")
    print(f"   Val   tiles   : {len(va)}")
    samp = np.load(tr[0] / "points.npy")
    print(f"   Sample tile   : {samp.shape}  dtype={samp.dtype}")
    del samp
elif DATA_SOURCE_MODE == "zip":
    import zipfile, collections
    gb = DATA_ZIP_PATH.stat().st_size / 1e9
    print(f"✅ Dataset (zip) : {DATA_ZIP_PATH}  ({gb:.2f} GB)")
    with zipfile.ZipFile(DATA_ZIP_PATH, "r") as z:
        pfx, tset, vset = collections.Counter(), set(), set()
        for info in z.infolist():
            parts = info.filename.strip("/").split("/")
            for sp, s in [("train", tset), ("val", vset)]:
                if sp in parts:
                    i = parts.index(sp)
                    if i + 1 < len(parts) and parts[i+1].startswith("tile_"):
                        s.add(parts[i+1]); pfx["/".join(parts[:i])] += 1
    ZIP_ROOT_PREFIX = pfx.most_common(1)[0][0] if pfx else ""
    print(f"   Train: {len(tset)}  Val: {len(vset)}  prefix='{ZIP_ROOT_PREFIX}'")
else:
    raise ValueError("Cannot find dataset. Set DATASET_ROOT or DATASET_ZIP_PATH above.")


In [ ]:
# ─── Cell 4  Global Configuration ────────────────────────────────────────────
# All tunable parameters in ONE place.
from pathlib import Path

# ── Stage plan (auto-promotes when metrics stabilise) ─────────────────────────
# Timing on P100 with 2048 pts, AMP, batch 6:
#   sanity ~2 min  |  medium ~40 min  |  full ~8.5 h  →  fits 10-h budget
STAGE_PLAN = [
    {"name":"sanity","epochs":10,"max_train":200,  "max_val":40,   "batch":6},
    {"name":"medium","epochs":20,"max_train":2000, "max_val":400,  "batch":6},
    {"name":"full",  "epochs":35,"max_train":14418,"max_val":3955, "batch":6},
]
START_PROFILE = "full"   # "sanity" | "medium" | "full"

if START_PROFILE not in [s["name"] for s in STAGE_PLAN]:
    raise ValueError(f"Unknown START_PROFILE: {START_PROFILE}")

idx0       = [s["name"] for s in STAGE_PLAN].index(START_PROFILE)
stage_plan = STAGE_PLAN[idx0:]

CFG = {
    # ── Data ──────────────────────────────────────────────────────────────────
    "use_zip"          : DATA_SOURCE_MODE == "zip",
    "zip_path"         : str(DATA_ZIP_PATH)  if DATA_ZIP_PATH  else "",
    "zip_prefix"       : ZIP_ROOT_PREFIX,
    "training_dir"     : str(TRAINING_ROOT)  if TRAINING_ROOT  else None,

    # ── Paths ─────────────────────────────────────────────────────────────────
    "feat_cache_dir"   : "/kaggle/working/feat_cache",
    "logs_dir"         : "/kaggle/working/logs",
    "model_path"       : "/kaggle/working/logs/best_model.pth",
    "ckpt_path"        : "/kaggle/working/logs/checkpoint.pth",
    "history_path"     : "/kaggle/working/logs/history.json",
    "curves_path"      : "/kaggle/working/logs/curves.png",
    "threshold_path"   : "/kaggle/working/logs/threshold.json",
    "dtm_dir"          : "/kaggle/working/dtm_output",
    "inference_cfg"    : "/kaggle/working/dtm_output/inference_config.json",

    # ── Training ──────────────────────────────────────────────────────────────
    "resume"           : True,        # auto-resume from checkpoint if it exists
    "max_pts"          : 2048,        # pts per tile (2048 => ~2x faster than 4096)
    "seed"             : 42,
    "stage_plan"       : stage_plan,

    # ── Optimiser ─────────────────────────────────────────────────────────────
    "max_lr"           : 0.01,        # OneCycleLR peak LR
    "weight_decay"     : 1e-4,
    "grad_clip"        : 5.0,

    # ── Loss ──────────────────────────────────────────────────────────────────
    "focal_alpha"      : 0.75,        # upweight ground class
    "focal_gamma"      : 2.0,

    # ── Stopping ──────────────────────────────────────────────────────────────
    "patience"         : 8,           # early-stop val checks without improvement
    "val_every"        : 2,           # run validation every N epochs

    # ── Stage promotion ───────────────────────────────────────────────────────
    "promo_window"     : 3,
    "promo_min_acc"    : 0.82,
    "promo_min_f1"     : 0.72,
    "promo_max_std"    : 0.02,
    "promo_min_eps"    : 3,

    # ── DataLoader (19 GB RAM budget) ──────────────────────────────────────────
    "workers"          : 2,
    "prefetch"         : 2,
    "use_amp"          : True,        # fp16 AMP (works on P100/sm_60)
    "grad_ckpt"        : True,        # recompute SA1+SA2 activations (~40% less VRAM)

    # ── DTM post-processing ────────────────────────────────────────────────────
    "dtm_resolution"   : 0.5,         # metres per pixel
    "dtm_smooth_sigma" : 1.0,         # Gaussian sigma for terrain smoothing
    "dtm_fill_method"  : "linear",    # griddata: 'linear' | 'nearest' | 'cubic'
}

# Init-stage values (overwritten by apply_stage during training)
_s0 = stage_plan[0]
CFG.update({"epochs": _s0["epochs"], "max_train": _s0["max_train"],
            "max_val": _s0["max_val"], "batch": _s0["batch"]})

for d in [CFG["logs_dir"], CFG["feat_cache_dir"], CFG["dtm_dir"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✅ Config ready")
print(f"   Start profile  : {START_PROFILE}")
print(f"   Init stage     : {_s0['name']}  ({_s0['max_train']} / {_s0['max_val']} tiles)")
print(f"   max_pts/tile   : {CFG['max_pts']}  (2048 for 10-h budget on P100)")
print(f"   batch_size     : {_s0['batch']}")
print(f"   AMP fp16       : {CFG['use_amp']}")
print(f"   Grad ckpt      : {CFG['grad_ckpt']}  (SA1+SA2, saves ~40% VRAM)")
print(f"   DTM resolution : {CFG['dtm_resolution']} m/px")
mem_report("after config")


In [ ]:
# ─── Cell 5  Geospatial Feature Cache ────────────────────────────────────────
# Computes 4 terrain features per point once, stores as float32 .npy files.
# Features: [delta_z, roughness, slope, density]  (all z-scored)
# Memory fix: float32 throughout (was float64 = double RAM usage).
import numpy as np
from pathlib import Path
from tqdm    import tqdm
import time, gc


def compute_features(xyz: np.ndarray) -> np.ndarray:
    """(N,3) float32 → (N,4) float32  [dz, rough, slope, dens] z-scored."""
    xyz   = xyz.astype(np.float32, copy=False)
    xmin, ymin = float(xyz[:,0].min()), float(xyz[:,1].min())
    xr = float(xyz[:,0].max() - xmin) + 1e-6
    yr = float(xyz[:,1].max() - ymin) + 1e-6
    GW = int(np.clip(xr / 2.0, 16, 64))
    GH = int(np.clip(yr / 2.0, 16, 64))
    NC = GH * GW

    xi = np.clip(((xyz[:,0] - xmin) / xr * GW).astype(np.int32), 0, GW-1)
    yi = np.clip(((xyz[:,1] - ymin) / yr * GH).astype(np.int32), 0, GH-1)
    ci = yi * GW + xi
    z  = xyz[:,2].copy()

    c_min = np.full(NC, np.inf,  dtype=np.float32)
    c_sum = np.zeros(NC,         dtype=np.float32)
    c_sq  = np.zeros(NC,         dtype=np.float32)
    c_cnt = np.zeros(NC,         dtype=np.float32)
    np.minimum.at(c_min, ci, z)
    np.add.at(c_sum, ci, z)
    np.add.at(c_sq,  ci, z*z)
    np.add.at(c_cnt, ci, 1.0)

    empty = (c_cnt == 0)
    zgm   = float(z.mean())
    c_cnt[empty]=1; c_min[empty]=zgm; c_sum[empty]=zgm; c_sq[empty]=zgm**2

    c_mean = c_sum / c_cnt
    c_std  = np.sqrt(np.maximum(c_sq/c_cnt - c_mean**2, 0.0))

    dz        = np.clip(z - c_min[ci], 0.0, None)
    roughness = c_std[ci].copy()
    dtm       = c_min.reshape(GH, GW).astype(np.float32)
    dy, dx    = np.gradient(dtm)
    slope     = np.sqrt(dx**2 + dy**2).ravel()[ci]
    density   = (c_cnt[ci] / (c_cnt.max() + 1e-6))

    feat = np.stack([dz, roughness, slope, density], axis=1).astype(np.float32)
    feat = (feat - feat.mean(0)) / (feat.std(0) + 1e-6)
    del c_min, c_sum, c_sq, c_cnt, c_mean, c_std, dtm, dx, dy
    return feat


def build_cache(cfg: dict, force: bool = False) -> None:
    """Pre-compute features for all tiles, save as float32 .npy."""
    if cfg["use_zip"]:
        print("⚠️  ZIP mode: features computed on-the-fly (no cache).")
        return

    root  = Path(cfg["training_dir"])
    croot = Path(cfg["feat_cache_dir"])
    built = skipped = errors = 0
    t0    = time.time()

    for split in ("train", "val"):
        sroot  = root / split
        scache = croot / split
        scache.mkdir(exist_ok=True)
        if not sroot.exists(): continue

        tiles = sorted([p.name for p in sroot.glob("tile_*")
                        if (p / "labels.npy").exists()])
        print(f"\n[{split}] {len(tiles)} tiles")

        for k, tile in enumerate(tqdm(tiles, desc=f"  {split}", ncols=80)):
            cp = scache / f"{tile}.npy"
            if cp.exists() and not force:
                skipped += 1; continue
            try:
                # mmap_mode="r": maps file without full RAM copy
                xyz  = np.load(sroot/tile/"points.npy",
                               mmap_mode="r")[:,:3].astype(np.float32)
                feat = compute_features(xyz)
                np.save(cp, feat)
                built += 1
                del xyz, feat
            except Exception as e:
                print(f"  ⚠️  {tile}: {e}"); errors += 1
            if k % 1000 == 0 and k > 0:
                gc.collect()
        gc.collect()

    print(f"\n✅ Cache: {built} built  {skipped} cached  {errors} errors"
          f"  ({time.time()-t0:.0f}s)")
    mem_report("after cache")


build_cache(CFG, force=False)


In [ ]:
# ─── Cell 6  Model: PointNet++ MSG + FocalLoss ────────────────────────────────
# Input : (B, N, 7)  [x,y,z, dz,roughness,slope,density]
# Output: (B, N, 2)  [non-ground logit, ground logit]
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_ckpt


# ── Geometric primitives ──────────────────────────────────────────────────────

def fps(xyz: torch.Tensor, k: int) -> torch.Tensor:
    """Farthest Point Sampling. Returns (B,k) indices."""
    B, N, _ = xyz.shape
    dev  = xyz.device
    idx  = torch.zeros(B, k, dtype=torch.long, device=dev)
    dist = torch.full((B, N), 1e10, device=dev)
    far  = torch.randint(0, N, (B,), device=dev)
    bi   = torch.arange(B, device=dev)
    for i in range(k):
        idx[:,i] = far
        c = xyz[bi, far].unsqueeze(1)
        d = ((xyz - c)**2).sum(-1)
        dist = torch.where(d < dist, d, dist)
        far  = dist.argmax(-1)
    return idx


def gather(pts: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    """pts (B,N,C), idx (B,...) → (B,...,C)."""
    B    = pts.shape[0]
    bi   = torch.arange(B, device=pts.device).view([B]+[1]*(idx.dim()-1)).expand_as(idx)
    return pts[bi, idx]


# ── Set Abstraction layers ────────────────────────────────────────────────────

class SA_MSG(nn.Module):
    """Multi-Scale Grouping Set Abstraction. One shared cdist for all radii."""
    def __init__(self, n_ctr, radii, samples, in_ch, mlp_specs):
        super().__init__()
        self.n_ctr   = n_ctr
        self.radii   = radii
        self.samples = samples
        self.mlps    = nn.ModuleList()
        for spec in mlp_specs:
            layers, last = [], in_ch + 3
            for d in spec:
                layers += [nn.Conv2d(last,d,1,bias=False), nn.BatchNorm2d(d), nn.ReLU(True)]
                last = d
            self.mlps.append(nn.Sequential(*layers))

    def forward(self, xyz, pts):
        fi   = fps(xyz, self.n_ctr)
        nxyz = gather(xyz, fi)               # (B,M,3)
        dist = torch.cdist(nxyz, xyz)        # (B,M,N) — computed once, reused

        outs = []
        for r, k, mlp in zip(self.radii, self.samples, self.mlps):
            msk = torch.where(dist <= r, dist, dist.new_full((), 1e10))
            idx = msk.topk(k, dim=-1, largest=False)[1]
            gxyz = gather(xyz, idx) - nxyz.unsqueeze(2)
            gpts = (torch.cat([gxyz, gather(pts, idx)], -1) if pts is not None else gxyz)
            feat = mlp(gpts.permute(0,3,2,1)).max(2)[0].permute(0,2,1)
            outs.append(feat)
            del gxyz, gpts, feat, idx, msk
        del dist
        return nxyz, torch.cat(outs, -1)


class SA(nn.Module):
    """Single-scale Set Abstraction (used for SA3 global layer)."""
    def __init__(self, n_ctr, radius, n_samp, in_ch, dims):
        super().__init__()
        self.n_ctr, self.radius, self.n_samp = n_ctr, radius, n_samp
        layers, last = [], in_ch + 3
        for d in dims:
            layers += [nn.Conv2d(last,d,1,bias=False), nn.BatchNorm2d(d), nn.ReLU(True)]
            last = d
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz, pts):
        fi   = fps(xyz, self.n_ctr)
        nxyz = gather(xyz, fi)
        dist = torch.cdist(nxyz, xyz)
        msk  = torch.where(dist <= self.radius, dist, dist.new_full((), 1e10))
        idx  = msk.topk(self.n_samp, dim=-1, largest=False)[1]
        gxyz = gather(xyz, idx) - nxyz.unsqueeze(2)
        gpts = (torch.cat([gxyz, gather(pts, idx)], -1) if pts is not None else gxyz)
        feat = self.mlp(gpts.permute(0,3,2,1)).max(2)[0].permute(0,2,1)
        del dist, msk, gxyz, gpts
        return nxyz, feat


class FP(nn.Module):
    """Feature Propagation (trilinear interpolation + MLP)."""
    def __init__(self, in_ch, dims):
        super().__init__()
        layers, last = [], in_ch
        for d in dims:
            layers += [nn.Conv1d(last,d,1,bias=False), nn.BatchNorm1d(d), nn.ReLU(True)]
            last = d
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz1, xyz2, p1, p2):
        dists, idx = torch.cdist(xyz1, xyz2).topk(3, dim=-1, largest=False)
        dists = torch.clamp(dists, 1e-10)
        w = 1.0 / dists; w = w / w.sum(-1, keepdim=True)
        ip = (gather(p2, idx) * w.unsqueeze(-1)).sum(2)
        feat = torch.cat([p1, ip], -1) if p1 is not None else ip
        return self.mlp(feat.permute(0,2,1)).permute(0,2,1)


# ── Main model ────────────────────────────────────────────────────────────────

class DTMNet(nn.Module):
    """
    PointNet++ MSG for DTM ground segmentation.
    Input  : (B,N,7) = [x,y,z, dz,roughness,slope,density]
    Output : (B,N,2) = [non-ground, ground]
    """
    FEAT = 4   # derived features (dz, roughness, slope, density)

    def __init__(self, num_cls=2, use_grad_ckpt=False):
        super().__init__()
        self.use_gc = use_grad_ckpt
        IC = self.FEAT

        # SA1: fine (r=0.5m) + medium (r=1.5m) → 64+128=192 ch
        self.sa1 = SA_MSG(512, [0.5,1.5], [32,64], IC, [[32,64],[64,128]])
        # SA2: coarse (r=3m) + very coarse (r=6m) → 128+256=384 ch
        self.sa2 = SA_MSG(128, [3.0,6.0], [64,128], 192, [[128,128],[128,256]])
        # SA3: global context
        self.sa3 = SA(32, 12.0, 128, 384, [256,512])
        # Feature propagation
        self.fp3 = FP(384+512, [512,256])
        self.fp2 = FP(192+256, [256,128])
        self.fp1 = FP(IC+128,  [128,128])
        # Classification head
        self.head = nn.Sequential(
            nn.Conv1d(128,128,1,bias=False), nn.BatchNorm1d(128), nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Conv1d(128, 64,1,bias=False), nn.BatchNorm1d(64),  nn.ReLU(True),
            nn.Conv1d( 64,num_cls,1),
        )

    def _sa1(self, xyz, p): return self.sa1(xyz, p)
    def _sa2(self, xyz, p): return self.sa2(xyz, p)

    def forward(self, x):
        xyz = x[:,:,:3].contiguous()
        l0  = x[:,:,3:].contiguous()

        if self.use_gc and self.training:
            l1xyz, l1 = grad_ckpt(self._sa1, xyz, l0,  use_reentrant=False)
            l2xyz, l2 = grad_ckpt(self._sa2, l1xyz, l1, use_reentrant=False)
        else:
            l1xyz, l1 = self.sa1(xyz, l0)
            l2xyz, l2 = self.sa2(l1xyz, l1)

        l3xyz, l3 = self.sa3(l2xyz, l2)
        l2 = self.fp3(l2xyz, l3xyz, l2, l3)
        l1 = self.fp2(l1xyz, l2xyz, l1, l2)
        l0 = self.fp1(xyz,   l1xyz, l0, l1)
        return self.head(l0.permute(0,2,1)).permute(0,2,1)


class FocalLoss(nn.Module):
    """Focal loss — fixes Precision/Recall imbalance from class weighting."""
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.a, self.g = alpha, gamma

    def forward(self, logits, tgt):
        ce = F.cross_entropy(logits, tgt, reduction="none")
        pt = torch.exp(-ce)
        at = torch.where(tgt==1,
                         logits.new_full(ce.shape, self.a),
                         logits.new_full(ce.shape, 1.0-self.a))
        return (at * (1.0-pt)**self.g * ce).mean()


def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

# Smoke-test
_m = DTMNet(use_grad_ckpt=True).to(device)
with torch.no_grad():
    _y = _m(torch.randn(2,64,7,device=device))
assert _y.shape == (2,64,2)
print(f"✅ DTMNet  params={n_params(_m):,}  output={_y.shape}")
del _m, _y; free_memory()
mem_report("after model def")


In [ ]:
# ─── Cell 7  Dataset Class ────────────────────────────────────────────────────
import io, zipfile
import numpy as np
from pathlib import Path, PurePosixPath
import torch
from torch.utils.data import Dataset


class DTMDataset(Dataset):
    """
    Yields (N,7) float32 features + (N,) int64 labels per tile.
    Features: [x, y, z, dz, roughness, slope, density]
    """
    def __init__(self, root_dir=None, zip_path=None, zip_prefix="",
                 feat_cache=None, split="train", augment=False,
                 max_pts=2048, max_tiles=None, seed=42):
        self.augment = augment; self.max_pts = max_pts
        self.split   = split;   self._zip   = None
        self.mode    = "zip" if zip_path else "dir"
        self.cache   = Path(feat_cache)/split if feat_cache else None

        if self.mode == "dir":
            self.root = Path(root_dir)
            if self.root.name in {"train","val"}: self.split = self.root.name
            self.tiles = [p.name for p in self.root.glob("tile_*")
                          if (p/"labels.npy").exists()]
        else:
            self.zip_path = Path(zip_path); self.pfx = zip_prefix.strip("/")
            self.tiles    = self._scan_zip()

        self.tiles = sorted(self.tiles)
        if max_tiles and len(self.tiles) > max_tiles:
            rng  = np.random.default_rng(seed)
            pick = rng.choice(len(self.tiles), max_tiles, replace=False)
            self.tiles = [self.tiles[i] for i in sorted(pick)]

        self._use_cache = False
        if self.mode == "dir" and self.cache and self.tiles:
            self._use_cache = (self.cache/f"{self.tiles[0]}.npy").exists()

        print(f"[{self.split}] {len(self.tiles)} tiles  "
              f"features={'✅ cache' if self._use_cache else '⚠️ on-the-fly'}")

    def _scan_zip(self):
        tiles = set()
        with zipfile.ZipFile(self.zip_path,"r") as zf:
            for info in zf.infolist():
                parts = PurePosixPath(info.filename).parts
                if self.split in parts:
                    i = parts.index(self.split)
                    if i+1 < len(parts) and parts[i+1].startswith("tile_"):
                        tiles.add(parts[i+1])
        return list(tiles)

    def _zmem(self, tile, fn):
        pre = f"{self.pfx}/" if self.pfx else ""
        return f"{pre}{self.split}/{tile}/{fn}"

    def _zopen(self):
        if self._zip is None: self._zip = zipfile.ZipFile(self.zip_path,"r")

    def _znpy(self, member):
        self._zopen()
        with self._zip.open(member) as f: return np.load(io.BytesIO(f.read()))

    def load_labels(self, idx):
        tile = self.tiles[idx]
        if self.mode == "dir":
            return np.load(self.root/tile/"labels.npy").astype(np.int64)
        return self._znpy(self._zmem(tile,"labels.npy")).astype(np.int64)

    def _load(self, tile):
        if self.mode == "dir":
            xyz = np.load(self.root/tile/"points.npy",
                          mmap_mode="r")[:,:3].astype(np.float32)
            lbl = np.load(self.root/tile/"labels.npy").astype(np.int64)
        else:
            xyz = self._znpy(self._zmem(tile,"points.npy")).astype(np.float32)[:,:3]
            lbl = self._znpy(self._zmem(tile,"labels.npy")).astype(np.int64)
        return xyz, lbl

    def __len__(self): return len(self.tiles)

    def __getitem__(self, idx):
        tile    = self.tiles[idx]
        xyz, lb = self._load(tile)
        feat4   = (np.load(self.cache/f"{tile}.npy") if self._use_cache
                   else compute_features(xyz))

        N      = xyz.shape[0]
        ch     = np.random.choice(N, self.max_pts, replace=(N < self.max_pts))
        xyz    = xyz[ch]; feat4 = feat4[ch]; lb = lb[ch]

        if self.augment:
            a  = np.random.uniform(0.0, 2*np.pi)
            c, s = np.cos(a).astype(np.float32), np.sin(a).astype(np.float32)
            R  = np.array([[c,-s,0],[s,c,0],[0,0,1]],np.float32)
            xyz = xyz @ R.T
            xyz += np.random.normal(0,.02,xyz.shape).astype(np.float32)
            xyz *= np.float32(np.random.uniform(.90,1.10))
            if np.random.rand()>.5: xyz[:,0] *= -1
            if np.random.rand()>.5: xyz[:,1] *= -1
            xyz[:,0] *= np.float32(np.random.uniform(.85,1.15))
            xyz[:,1] *= np.float32(np.random.uniform(.85,1.15))
            xyz[:,2] *= np.float32(np.random.uniform(.80,1.20))
            if np.random.rand()<.3:
                kn  = int(self.max_pts * np.random.uniform(.75,1.))
                k   = np.random.choice(self.max_pts, kn, replace=False)
                rf  = np.random.choice(k, self.max_pts-kn, replace=True)
                o   = np.concatenate([k,rf])
                xyz=xyz[o]; feat4=feat4[o]; lb=lb[o]

        xyz[:,0] -= xyz[:,0].mean(); xyz[:,1] -= xyz[:,1].mean()
        feat = np.concatenate([xyz, feat4], 1).astype(np.float32)
        return torch.from_numpy(feat), torch.from_numpy(lb)

    def __del__(self):
        if getattr(self,"_zip",None):
            try: self._zip.close()
            except: pass


In [ ]:
# ─── Cell 8  Training Loop ────────────────────────────────────────────────────
# Focal Loss + OneCycleLR + AMP (torch.cuda.amp.*) + staged auto-promotion
import time, json
import torch, torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np, matplotlib.pyplot as plt
from tqdm import tqdm


def _loaders(cfg):
    kw = dict(feat_cache=cfg["feat_cache_dir"], max_pts=cfg["max_pts"],
              seed=cfg["seed"])
    if cfg["use_zip"]:
        z = dict(zip_path=cfg["zip_path"], zip_prefix=cfg["zip_prefix"])
        tr = DTMDataset(**z,**kw, split="train", augment=True,  max_tiles=cfg["max_train"])
        va = DTMDataset(**z,**kw, split="val",   augment=False, max_tiles=cfg["max_val"])
    else:
        td = cfg["training_dir"]
        tr = DTMDataset(root_dir=td+"/train",**kw, augment=True,  max_tiles=cfg["max_train"])
        va = DTMDataset(root_dir=td+"/val",  **kw, augment=False, max_tiles=cfg["max_val"])

    nw, pf = cfg["workers"], cfg["prefetch"]
    base = dict(num_workers=nw, pin_memory=True,
                persistent_workers=(nw>0),
                prefetch_factor=(pf if nw>0 else None))
    tl = DataLoader(tr, batch_size=cfg["batch"], shuffle=True,  drop_last=True,  **base)
    vl = DataLoader(va, batch_size=cfg["batch"], shuffle=False, drop_last=False, **base)
    return tr, va, tl, vl


def _should_promote(hist, stage, cfg):
    win = cfg["promo_window"]; mep = cfg["promo_min_eps"]
    sh  = [h for h in hist if h.get("stage")==stage
           and not np.isnan(h.get("va", float("nan")))]
    if len(sh) < max(win, mep): return False
    v = np.array([h["va"] for h in sh[-win:]])
    f = np.array([h["f1"] for h in sh[-win:]])
    return (v.mean() >= cfg["promo_min_acc"]
            and f.mean() >= cfg["promo_min_f1"]
            and v.std()  <= cfg["promo_max_std"])


def _apply_stage(cfg, plan, idx):
    s = plan[idx]
    cfg["max_train"]=s["max_train"]; cfg["max_val"]=s["max_val"]
    cfg["batch"]=s["batch"];         cfg["epochs"]=s["epochs"]
    return s["name"]


def _save_curves(hist, path):
    vh = [h for h in hist if not np.isnan(h.get("vl", float("nan")))]
    ep = [h["ep"] for h in hist]; ev = [h["ep"] for h in vh]
    fig, ax = plt.subplots(1,3,figsize=(16,4))
    ax[0].plot(ep, [h["tl"] for h in hist], label="Train")
    if vh: ax[0].plot(ev,[h["vl"] for h in vh],label="Val")
    ax[0].set(title="Loss",xlabel="Epoch"); ax[0].legend(); ax[0].grid(True)
    ax[1].plot(ep,[h["ta"] for h in hist],label="Train acc")
    if vh: ax[1].plot(ev,[h["va"] for h in vh],label="Val acc")
    ax[1].axhline(.95,ls="--",color="red",alpha=.4,label="95% target")
    ax[1].set(title="Accuracy",xlabel="Epoch",ylim=(.75,1.)); ax[1].legend(); ax[1].grid(True)
    if vh:
        ax[2].plot(ev,[h["p"] for h in vh],label="Precision")
        ax[2].plot(ev,[h["r"] for h in vh],label="Recall")
        ax[2].plot(ev,[h["f1"] for h in vh],label="F1",lw=2)
    ax[2].set(title="P/R/F1",xlabel="Epoch"); ax[2].legend(); ax[2].grid(True)
    plt.suptitle("DTMNet Training — Kaggle P100",fontsize=13)
    plt.tight_layout(); plt.savefig(path,dpi=150,bbox_inches="tight")
    plt.show(); plt.close(fig)


def train(cfg, device):
    torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])

    model = DTMNet(num_cls=2, use_grad_ckpt=cfg["grad_ckpt"]).to(device)
    print(f"Model  params : {n_params(model):,}")

    crit  = FocalLoss(cfg["focal_alpha"], cfg["focal_gamma"])
    opt   = torch.optim.AdamW(model.parameters(),
                               lr=cfg["max_lr"]/25, weight_decay=cfg["weight_decay"],
                               betas=(0.9,0.999))
    use_amp = cfg["use_amp"]
    scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)

    plan  = cfg["stage_plan"]; sidx = 0
    bva   = 0.0; pat = 0; hist = []; ep0 = 1

    # ── Resume ────────────────────────────────────────────────────────────────
    ckpt = cfg["ckpt_path"]
    if cfg["resume"] and Path(ckpt).exists():
        c = torch.load(ckpt, map_location=device)
        model.load_state_dict(c["model"])
        try: opt.load_state_dict(c["opt"])
        except: pass
        try: scaler.load_state_dict(c["scaler"])
        except: pass
        ep0  = c.get("ep",0)+1; bva = c.get("bva",0.); pat = c.get("pat",0)
        hist = c.get("hist",[]); sidx = c.get("sidx",0)
        _apply_stage(cfg, plan, sidx)
        print(f"✅ Resumed epoch {ep0}  best_val={bva:.4f}  stage={plan[sidx]['name']}")
    else:
        _apply_stage(cfg, plan, 0)
        print("ℹ️  Training from scratch")

    stage = plan[sidx]["name"]
    tr_ds, va_ds, tr_ld, va_ld = _loaders(cfg)

    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=cfg["max_lr"], epochs=cfg["epochs"],
        steps_per_epoch=len(tr_ld), pct_start=.30,
        anneal_strategy="cos", div_factor=25., final_div_factor=1000.)
    if cfg["resume"] and Path(ckpt).exists():
        c2 = torch.load(ckpt, map_location=device)
        if "sched" in c2:
            try: sched.load_state_dict(c2["sched"])
            except: pass

    ve = max(1, cfg["val_every"])
    print(f"\n▶ Stage={stage} | AMP={use_amp} | gc={cfg['grad_ckpt']} | val_every={ve}")
    print(f"  Batches: train={len(tr_ld)}  val={len(va_ld)}")
    mem_report("train start")

    for ep in range(ep0, cfg["epochs"]+1):
        t0 = time.time()
        model.train(); tl_sum=tok=tn=0

        for feat, lbl in tqdm(tr_ld, desc=f"Ep{ep:03d} Train", leave=False, ncols=80):
            feat = feat.to(device, non_blocking=True)
            lbl  = lbl.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(feat); B,N,C = out.shape
                loss = crit(out.reshape(B*N,C), lbl.reshape(B*N))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(opt); scaler.update(); sched.step()
            with torch.no_grad():
                p = out.argmax(-1); tok += (p==lbl).sum().item()
                tn += B*N; tl_sum += loss.item()*B

        ta = tok/max(1,tn); tl = tl_sum/max(1,len(tr_ds))
        lr = sched.get_last_lr()[0]

        # ── Validation ────────────────────────────────────────────────────────
        rv = (ep%ve==0) or (ep==cfg["epochs"])
        vl=va=pr=rc=f1=float("nan")
        if rv:
            model.eval(); vls=vok=vn=tp=fp=fn=0
            with torch.no_grad():
                for feat,lbl in tqdm(va_ld,desc=f"Ep{ep:03d} Val",leave=False,ncols=80):
                    feat=feat.to(device,non_blocking=True)
                    lbl=lbl.to(device,non_blocking=True)
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        out=model(feat); B,N,C=out.shape
                        loss=crit(out.reshape(B*N,C),lbl.reshape(B*N))
                    p=out.argmax(-1)
                    vls+=loss.item()*B; vok+=(p==lbl).sum().item(); vn+=B*N
                    tp+=((p==1)&(lbl==1)).sum().item()
                    fp+=((p==1)&(lbl==0)).sum().item()
                    fn+=((p==0)&(lbl==1)).sum().item()
            vl=vls/max(1,len(va_ds)); va=vok/max(1,vn)
            pr=tp/(tp+fp+1e-9); rc=tp/(tp+fn+1e-9)
            f1=2*pr*rc/(pr+rc+1e-9)
            free_memory()

        el = time.time()-t0
        flag = "  *** BEST ***" if rv and va>bva else ""
        if rv:
            print(f"Ep{ep:3d}/{cfg['epochs']} [{stage}] "
                  f"| T {tl:.4f}/{ta:.4f} | V {vl:.4f}/{va:.4f} "
                  f"| P{pr:.3f} R{rc:.3f} F1{f1:.3f} | lr{lr:.1e} {el:.0f}s{flag}")
        else:
            print(f"Ep{ep:3d}/{cfg['epochs']} [{stage}] "
                  f"| T {tl:.4f}/{ta:.4f} | val_skip | lr{lr:.1e} {el:.0f}s")

        hist.append(dict(ep=ep,stage=stage,tl=tl,ta=ta,vl=vl,
                         va=va if rv else (hist[-1]["va"] if hist else 0.),
                         p=pr,r=rc,f1=f1 if rv else (hist[-1]["f1"] if hist else 0.),lr=lr))

        if rv:
            if va>bva: bva=va; pat=0; torch.save(model.state_dict(), cfg["model_path"])
            else: pat+=1

        torch.save(dict(ep=ep,bva=bva,pat=pat,sidx=sidx,hist=hist,
                        model=model.state_dict(),opt=opt.state_dict(),
                        sched=sched.state_dict(),scaler=scaler.state_dict()),
                   cfg["ckpt_path"])

        # ── Auto-stage promotion ───────────────────────────────────────────────
        if rv and cfg.get("auto_promote",True) and sidx+1<len(plan):
            if _should_promote(hist, stage, cfg):
                sidx+=1; stage=_apply_stage(cfg, plan, sidx)
                print(f"\n🚀 Stage → '{stage}'  "
                      f"({cfg['max_train']}/{cfg['max_val']} tiles, batch={cfg['batch']})")
                free_memory()
                tr_ds,va_ds,tr_ld,va_ld = _loaders(cfg)
                sched = torch.optim.lr_scheduler.OneCycleLR(
                    opt, max_lr=cfg["max_lr"], epochs=cfg["epochs"]-ep,
                    steps_per_epoch=len(tr_ld), pct_start=.10,
                    anneal_strategy="cos", div_factor=10., final_div_factor=1000.)
                pat=0; mem_report(f"stage→{stage}")

        if rv and pat >= cfg["patience"]:
            print(f"\n  Early stop at epoch {ep} ({pat} checks without improvement)")
            break

    with open(cfg["history_path"],"w") as f: json.dump(hist,f,indent=2)
    _save_curves(hist, cfg["curves_path"])
    print(f"\n✅ Training done.  Best val acc: {bva:.4f}")
    free_memory(); mem_report("train done")
    return model, hist, bva


# ── default auto_promote to True ─────────────────────────────────────────────
CFG.setdefault("auto_promote", True)

trained_model, history, best_acc = train(CFG, device)


In [ ]:
# ─── Cell 9  Threshold Optimisation ─────────────────────────────────────────
import json
import numpy as np, torch, torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt


def find_threshold(model, val_loader, device, use_amp=True,
                   thresholds=None):
    """Sweep decision thresholds; return the one maximising ground-class F1."""
    if thresholds is None:
        thresholds = np.arange(0.20, 0.85, 0.01)

    model.eval()
    probs_list, lbls_list = [], []
    print("Threshold sweep — full validation pass...")
    with torch.no_grad():
        for feat, lbl in tqdm(val_loader, ncols=80):
            feat = feat.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=(use_amp and device.type=="cuda")):
                out = model(feat)
            probs = F.softmax(out, -1)[:,:,1]
            probs_list.append(probs.reshape(-1).float().cpu().numpy())
            lbls_list.append(lbl.reshape(-1).numpy())
    free_memory()

    probs = np.concatenate(probs_list).astype(np.float32)
    lbls  = np.concatenate(lbls_list).astype(np.int32)

    best_t = 0.5; best_f1 = 0.; best_row = {}; rows = []
    for t in thresholds:
        p  = (probs >= t).astype(np.int32)
        tp = int(((p==1)&(lbls==1)).sum())
        fp = int(((p==1)&(lbls==0)).sum())
        fn = int(((p==0)&(lbls==1)).sum())
        tn = int(((p==0)&(lbls==0)).sum())
        pr = tp/(tp+fp+1e-9); rc = tp/(tp+fn+1e-9)
        f1 = 2*pr*rc/(pr+rc+1e-9)
        acc = (tp+tn)/(len(lbls)+1e-9)
        rows.append(dict(t=round(float(t),2),acc=acc,p=pr,r=rc,f1=f1))
        if f1 > best_f1: best_f1=f1; best_t=float(t); best_row=rows[-1]

    print(f"\n🎯 Optimal threshold : {best_t:.2f}")
    print(f"   Accuracy : {best_row['acc']:.4f}")
    print(f"   Precision: {best_row['p']:.4f}")
    print(f"   Recall   : {best_row['r']:.4f}")
    print(f"   F1       : {best_row['f1']:.4f}")

    fig, ax = plt.subplots(figsize=(9,4))
    ts  = [r["t"] for r in rows]
    ax.plot(ts, [r["acc"] for r in rows], label="Accuracy", lw=2)
    ax.plot(ts, [r["f1"]  for r in rows], label="F1 ground", lw=2)
    ax.axvline(best_t, ls="--", color="red",  label=f"Best={best_t:.2f}")
    ax.axhline(.95,    ls=":",  color="gray", alpha=.5)
    ax.set(title="Threshold vs Accuracy / F1", xlabel="Threshold")
    ax.legend(); ax.grid(True); plt.tight_layout()
    plt.savefig(CFG["logs_dir"]+"/threshold_curve.png", dpi=150)
    plt.show(); plt.close(fig)
    return best_t, best_row, rows


# Load best weights (no grad_ckpt at inference)
best_model = DTMNet(num_cls=2, use_grad_ckpt=False).to(device)
best_model.load_state_dict(torch.load(CFG["model_path"], map_location=device))
print("✅ Best model weights loaded")

# Rebuild val loader (full val set for accurate threshold)
CFG_FULL = {**CFG, "max_val": 3955}
if CFG["use_zip"]:
    va_ds_full = DTMDataset(zip_path=CFG["zip_path"],zip_prefix=CFG["zip_prefix"],
                            feat_cache=CFG["feat_cache_dir"],max_pts=CFG["max_pts"],
                            split="val",augment=False,max_tiles=CFG_FULL["max_val"])
else:
    va_ds_full = DTMDataset(root_dir=CFG["training_dir"]+"/val",
                            feat_cache=CFG["feat_cache_dir"],max_pts=CFG["max_pts"],
                            augment=False,max_tiles=CFG_FULL["max_val"])

va_ld_full = __import__("torch").utils.data.DataLoader(
    va_ds_full, batch_size=CFG["batch"], shuffle=False, num_workers=CFG["workers"],
    pin_memory=True, prefetch_factor=CFG["prefetch"] if CFG["workers"]>0 else None,
    persistent_workers=(CFG["workers"]>0))

OPT_THRESH, OPT_METRICS, THRESH_ROWS = find_threshold(
    best_model, va_ld_full, device, use_amp=CFG["use_amp"])

with open(CFG["threshold_path"],"w") as f:
    json.dump(dict(optimal_threshold=OPT_THRESH,
                   metrics=OPT_METRICS, sweep=THRESH_ROWS), f, indent=2)

print(f"\n💾 Threshold saved → {CFG['threshold_path']}")
print("\n" + "="*60)
print("  CLASSIFICATION RESULTS")
print("="*60)
print(f"  Best val acc (0.50) : {best_acc:.4f}")
print(f"  Acc  at {OPT_THRESH:.2f}       : {OPT_METRICS['acc']:.4f}")
print(f"  Precision           : {OPT_METRICS['p']:.4f}")
print(f"  Recall              : {OPT_METRICS['r']:.4f}")
print(f"  F1 (ground class)   : {OPT_METRICS['f1']:.4f}")
met = OPT_METRICS["acc"] >= .95
print(f"\n  95% target: {'✅ YES' if met else '❌ Not yet'}")
print("="*60)
mem_report("after threshold")


In [ ]:
# ─── Cell 10  DTM Post-Processing ────────────────────────────────────────────
# Converts the ground-point segmentation into a clean, usable DTM:
#   1. Run model → extract predicted ground XYZ per tile
#   2. Rasterize to regular grid (min-elevation per cell)
#   3. Fill holes / nodata gaps (scipy.interpolate.griddata)
#   4. Fill pits (morphological grey-closing)
#   5. Gaussian smoothing (removes residual noise)
#   6. Save per-tile DTM as .npy + mosaic if tiles are geo-referenced
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np, torch, torch.nn.functional as F
from pathlib import Path
from scipy   import ndimage, interpolate
from tqdm    import tqdm
import matplotlib.pyplot as plt, json, gc


# ── Helpers ───────────────────────────────────────────────────────────────────

def rasterize(xyz_ground: np.ndarray, res: float = 0.5):
    """
    Convert ground point cloud to regular elevation grid.

    Strategy: for each cell take the lowest-20th-percentile Z of contained
    points (more robust than pure minimum against residual mis-classifications).

    Returns
    -------
    dtm   : (H, W) float32    — elevation in metres; NaN where no points
    mask  : (H, W) bool       — True where data exists
    geo   : (xmin, ymin, res) — for georeferencing
    """
    if len(xyz_ground) == 0:
        return None, None, None

    xmin, ymin = xyz_ground[:,0].min(), xyz_ground[:,1].min()
    xmax, ymax = xyz_ground[:,0].max(), xyz_ground[:,1].max()

    W = max(2, int(np.ceil((xmax - xmin) / res)) + 1)
    H = max(2, int(np.ceil((ymax - ymin) / res)) + 1)

    xi = np.clip(((xyz_ground[:,0]-xmin)/res).astype(np.int32), 0, W-1)
    yi = np.clip(((xyz_ground[:,1]-ymin)/res).astype(np.int32), 0, H-1)

    # Accumulate per-cell: collect all Z values, pick robust-min (20th percentile)
    from collections import defaultdict
    cell_z = defaultdict(list)
    for x,y,z in zip(xi,yi,xyz_ground[:,2]):
        cell_z[(int(x),int(y))].append(z)

    dtm  = np.full((H, W), np.nan, dtype=np.float32)
    for (cx,cy), zs in cell_z.items():
        dtm[cy, cx] = np.percentile(zs, 20)   # robust ground estimate

    mask = ~np.isnan(dtm)
    return dtm, mask, (float(xmin), float(ymin), res)


def fill_holes(dtm: np.ndarray, mask: np.ndarray, method: str = "linear"):
    """
    Interpolate over NaN cells using scipy.interpolate.griddata.
    Falls back to nearest-neighbor for points outside convex hull.
    """
    if mask.sum() < 4 or (~mask).sum() == 0:
        return dtm.copy()

    ys, xs = np.where(mask)
    vals   = dtm[ys, xs]

    all_y, all_x = np.mgrid[0:dtm.shape[0], 0:dtm.shape[1]]
    pts   = np.column_stack([xs, ys])
    qpts  = np.column_stack([all_x.ravel(), all_y.ravel()])

    try:
        filled_lin = interpolate.griddata(pts, vals, qpts, method=method,
                                          fill_value=np.nan)
    except Exception:
        filled_lin = np.full(len(qpts), np.nan)

    # Fill any remaining NaN (outside convex hull) with nearest
    nan_mask_flat = np.isnan(filled_lin)
    if nan_mask_flat.any():
        filled_nn  = interpolate.griddata(pts, vals, qpts, method="nearest")
        filled_lin[nan_mask_flat] = filled_nn[nan_mask_flat]

    return filled_lin.reshape(dtm.shape).astype(np.float32)


def fill_pits(dtm: np.ndarray, iterations: int = 2) -> np.ndarray:
    """
    Remove artificial pits (small negative artifacts from mis-classified points).
    Uses morphological grey-closing: erosion then dilation.
    """
    out = dtm.copy()
    for _ in range(iterations):
        # Closing fills dark holes without expanding bright peaks
        out = ndimage.grey_closing(out, size=3)
    return out.astype(np.float32)


def smooth_terrain(dtm: np.ndarray, sigma: float = 1.0) -> np.ndarray:
    """Gaussian smoothing — removes high-freq noise, preserves terrain shape."""
    return ndimage.gaussian_filter(dtm, sigma=sigma).astype(np.float32)


def tile_to_dtm(xyz_ground: np.ndarray, res: float, fill_method: str,
                smooth_sigma: float) -> dict:
    """Full pipeline for one tile. Returns dict with DTM arrays + metadata."""
    if xyz_ground is None or len(xyz_ground) < 10:
        return None

    dtm_raw, mask, geo = rasterize(xyz_ground, res)
    if dtm_raw is None:
        return None

    dtm_filled   = fill_holes(dtm_raw, mask, method=fill_method)
    dtm_nopits   = fill_pits(dtm_filled)
    dtm_smooth   = smooth_terrain(dtm_nopits, sigma=smooth_sigma)

    return dict(
        raw      = dtm_raw,
        filled   = dtm_filled,
        no_pits  = dtm_nopits,
        clean    = dtm_smooth,      # final product
        geo      = geo,             # (xmin, ymin, res)
        n_ground = len(xyz_ground),
    )


# ── Inference + DTM generation ────────────────────────────────────────────────

def generate_dtm(model, cfg, device, threshold: float,
                 n_tiles: int = None, split: str = "val"):
    """
    Run model on all tiles of a split, extract ground points,
    generate per-tile DTMs, save as .npy files.
    """
    model.eval()
    dtm_dir = Path(cfg["dtm_dir"]) / split
    dtm_dir.mkdir(parents=True, exist_ok=True)

    use_amp = cfg["use_amp"]
    res     = cfg["dtm_resolution"]
    sm_sig  = cfg["dtm_smooth_sigma"]
    fmeth   = cfg["dtm_fill_method"]

    # Build dataset WITHOUT augmentation for inference
    kw = dict(feat_cache=cfg["feat_cache_dir"], max_pts=cfg["max_pts"])
    if cfg["use_zip"]:
        ds = DTMDataset(zip_path=cfg["zip_path"], zip_prefix=cfg["zip_prefix"],
                        **kw, split=split, augment=False,
                        max_tiles=n_tiles or 99999)
    else:
        ds = DTMDataset(root_dir=cfg["training_dir"]+f"/{split}",
                        **kw, augment=False, max_tiles=n_tiles or 99999)

    print(f"Generating DTMs for {len(ds)} tiles [{split}]  "
          f"(threshold={threshold:.2f}, res={res}m)...")

    stats = dict(tiles=0, ground_pts=0, skipped=0)

    for idx in tqdm(range(len(ds)), ncols=80):
        tile_name = ds.tiles[idx]
        out_path  = dtm_dir / f"{tile_name}_dtm.npy"
        meta_path = dtm_dir / f"{tile_name}_meta.json"
        if out_path.exists():
            stats["tiles"] += 1; continue

        feat, lbl_gt = ds[idx]
        feat_gpu = feat.unsqueeze(0).to(device)

        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=(use_amp and device.type=="cuda")):
                logits = model(feat_gpu)                # (1,N,2)
            prob_ground = F.softmax(logits, -1)[0,:,1].cpu().numpy()

        # Predicted ground: probability > threshold
        ground_mask  = prob_ground >= threshold
        xyz_all      = feat.numpy()[:,:3]               # (N,3) subsampled
        xyz_ground   = xyz_all[ground_mask]

        if len(xyz_ground) < 10:
            stats["skipped"] += 1; continue

        result = tile_to_dtm(xyz_ground, res, fmeth, sm_sig)
        if result is None:
            stats["skipped"] += 1; continue

        np.save(out_path, result["clean"])
        with open(meta_path,"w") as f:
            json.dump(dict(tile=tile_name,
                           geo=result["geo"],
                           shape=list(result["clean"].shape),
                           n_ground=int(result["n_ground"]),
                           threshold=threshold,
                           resolution=res), f, indent=2)
        stats["tiles"]       += 1
        stats["ground_pts"]  += len(xyz_ground)
        del feat_gpu, logits, prob_ground, result
        if idx % 200 == 0 and idx > 0:
            free_memory()

    free_memory()
    print(f"\n✅ DTM generation complete")
    print(f"   Tiles processed : {stats['tiles']}")
    print(f"   Tiles skipped   : {stats['skipped']}")
    print(f"   Total gnd pts   : {stats['ground_pts']:,}")
    print(f"   Saved to        : {dtm_dir}")
    return dtm_dir


print("Running DTM generation on validation tiles...")
DTM_DIR = generate_dtm(best_model, CFG, device, threshold=OPT_THRESH,
                        n_tiles=None, split="val")
mem_report("after DTM generation")


In [ ]:
# ─── Cell 11  Visualisation + Export ─────────────────────────────────────────
import numpy as np, matplotlib.pyplot as plt, json
from pathlib import Path


def visualise_dtm_tiles(dtm_dir, n_samples=6, save_path=None):
    """Plot a grid of sample DTM tiles side-by-side."""
    files = sorted(Path(dtm_dir).glob("*_dtm.npy"))[:n_samples]
    if not files:
        print("No DTM files found."); return

    cols = min(3, len(files)); rows = (len(files)+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
    axes = np.array(axes).ravel() if hasattr(axes,"ravel") else [axes]

    for i, fp in enumerate(files):
        dtm = np.load(fp)
        im  = axes[i].imshow(dtm, cmap="terrain", origin="lower",
                              interpolation="bilinear")
        plt.colorbar(im, ax=axes[i], shrink=.8, label="Elev (m)")
        axes[i].set_title(fp.stem.replace("_dtm",""), fontsize=9)
        axes[i].axis("off")

    for j in range(len(files), len(axes)):
        axes[j].axis("off")

    plt.suptitle("Generated DTM Tiles — AI Ground Segmentation", fontsize=13)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show(); plt.close(fig)


# Show sample DTMs
visualise_dtm_tiles(DTM_DIR, n_samples=6,
                    save_path=CFG["logs_dir"]+"/sample_dtm_tiles.png")


# ── Write inference config (for web interface integration) ────────────────────
import json, torch

inference_cfg = {
    "model_arch"      : "DTMNet",
    "model_params"    : {"num_cls": 2, "use_grad_ckpt": False},
    "n_input_features": 7,
    "feature_order"   : ["x", "y", "z", "delta_z", "roughness", "slope", "density"],
    "max_pts_per_tile": CFG["max_pts"],
    "optimal_threshold": OPT_THRESH,
    "dtm_resolution"  : CFG["dtm_resolution"],
    "dtm_smooth_sigma": CFG["dtm_smooth_sigma"],
    "dtm_fill_method" : CFG["dtm_fill_method"],
    "classification_metrics": OPT_METRICS,
    "pytorch_version" : torch.__version__,
}
with open(CFG["inference_cfg"], "w") as f:
    json.dump(inference_cfg, f, indent=2)
print(f"✅ inference_config.json saved → {CFG['inference_cfg']}")


# ── Write standalone inference script ────────────────────────────────────────
INFER_SCRIPT = '''\
"""
dtm_inference.py — standalone script to run DTMNet on a new point cloud tile.
Usage:
    python dtm_inference.py --points tile/points.npy --out dtm_output.npy \\
                            --model best_model.pth   --config inference_config.json
"""
import argparse, json, numpy as np
import torch, torch.nn.functional as F
from pathlib import Path
from scipy   import ndimage, interpolate
from collections import defaultdict


def compute_features(xyz):
    xyz = xyz.astype(np.float32)
    xmin,ymin = xyz[:,0].min(), xyz[:,1].min()
    xr = float(xyz[:,0].max()-xmin)+1e-6; yr = float(xyz[:,1].max()-ymin)+1e-6
    GW = int(np.clip(xr/2.,16,64)); GH = int(np.clip(yr/2.,16,64)); NC=GH*GW
    xi = np.clip(((xyz[:,0]-xmin)/xr*GW).astype(np.int32),0,GW-1)
    yi = np.clip(((xyz[:,1]-ymin)/yr*GH).astype(np.int32),0,GH-1)
    ci = yi*GW+xi; z = xyz[:,2].copy()
    c_min=np.full(NC,np.inf,dtype=np.float32); c_sum=np.zeros(NC,dtype=np.float32)
    c_sq=np.zeros(NC,dtype=np.float32);        c_cnt=np.zeros(NC,dtype=np.float32)
    np.minimum.at(c_min,ci,z); np.add.at(c_sum,ci,z)
    np.add.at(c_sq,ci,z*z);    np.add.at(c_cnt,ci,1.)
    e=c_cnt==0; zgm=float(z.mean())
    c_cnt[e]=1; c_min[e]=zgm; c_sum[e]=zgm; c_sq[e]=zgm**2
    c_mean=c_sum/c_cnt; c_std=np.sqrt(np.maximum(c_sq/c_cnt-c_mean**2,0.))
    dz=np.clip(z-c_min[ci],0.,None); rough=c_std[ci].copy()
    dtm=c_min.reshape(GH,GW).astype(np.float32); dy,dx=np.gradient(dtm)
    slope=np.sqrt(dx**2+dy**2).ravel()[ci]; dens=(c_cnt[ci]/(c_cnt.max()+1e-6))
    feat=np.stack([dz,rough,slope,dens],1).astype(np.float32)
    feat=(feat-feat.mean(0))/(feat.std(0)+1e-6)
    return feat


def rasterize(xyz,res):
    xmin,ymin=xyz[:,0].min(),xyz[:,1].min()
    xmax,ymax=xyz[:,0].max(),xyz[:,1].max()
    W=max(2,int(np.ceil((xmax-xmin)/res))+1); H=max(2,int(np.ceil((ymax-ymin)/res))+1)
    xi=np.clip(((xyz[:,0]-xmin)/res).astype(np.int32),0,W-1)
    yi=np.clip(((xyz[:,1]-ymin)/res).astype(np.int32),0,H-1)
    cells=defaultdict(list)
    for x,y,z in zip(xi,yi,xyz[:,2]): cells[(x,y)].append(z)
    dtm=np.full((H,W),np.nan,dtype=np.float32)
    for (cx,cy),zs in cells.items(): dtm[cy,cx]=np.percentile(zs,20)
    return dtm,(float(xmin),float(ymin),res)


def fill_and_smooth(dtm,fill_method="linear",sigma=1.0):
    mask=~np.isnan(dtm)
    if mask.sum()<4: return dtm
    ys,xs=np.where(mask); vals=dtm[ys,xs]
    ay,ax=np.mgrid[0:dtm.shape[0],0:dtm.shape[1]]
    pts=np.column_stack([xs,ys]); qpts=np.column_stack([ax.ravel(),ay.ravel()])
    try: filled=interpolate.griddata(pts,vals,qpts,method=fill_method,fill_value=np.nan)
    except: filled=np.full(len(qpts),np.nan)
    nan_m=np.isnan(filled)
    if nan_m.any(): filled[nan_m]=interpolate.griddata(pts,vals,qpts,method="nearest")[nan_m]
    out=filled.reshape(dtm.shape).astype(np.float32)
    out=ndimage.grey_closing(out,size=3)
    out=ndimage.gaussian_filter(out,sigma=sigma)
    return out.astype(np.float32)


# ---- Model (copy DTMNet definition here or import from your package) ----
# For brevity this script assumes you import DTMNet from a companion module.
# Replace the import below with the full class definition if running standalone.
try:
    from dtm_model import DTMNet
except ImportError:
    raise ImportError("Put the DTMNet class definition in dtm_model.py next to this script.")


def main():
    ap=argparse.ArgumentParser()
    ap.add_argument("--points",required=True); ap.add_argument("--out",required=True)
    ap.add_argument("--model",required=True);  ap.add_argument("--config",required=True)
    ap.add_argument("--max_pts",type=int,default=2048)
    args=ap.parse_args()

    with open(args.config) as f: cfg=json.load(f)

    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model=DTMNet(**cfg["model_params"]).to(device)
    model.load_state_dict(torch.load(args.model,map_location=device))
    model.eval()

    xyz=np.load(args.points).astype(np.float32)[:,:3]
    feat4=compute_features(xyz)
    N=xyz.shape[0]
    if N>args.max_pts:
        ch=np.random.choice(N,args.max_pts,replace=False)
        xyz=xyz[ch]; feat4=feat4[ch]
    xyz_n=xyz.copy(); xyz_n[:,0]-=xyz_n[:,0].mean(); xyz_n[:,1]-=xyz_n[:,1].mean()
    feat=np.concatenate([xyz_n,feat4],1).astype(np.float32)

    with torch.no_grad():
        logits=model(torch.from_numpy(feat).unsqueeze(0).to(device))
        prob=F.softmax(logits,-1)[0,:,1].cpu().numpy()

    threshold=cfg.get("optimal_threshold",0.5)
    gnd=xyz[prob>=threshold]
    if len(gnd)<10: print("WARNING: very few ground points detected"); return

    dtm_raw,geo=rasterize(gnd,cfg["dtm_resolution"])
    dtm_clean=fill_and_smooth(dtm_raw,cfg["dtm_fill_method"],cfg["dtm_smooth_sigma"])
    np.save(args.out,dtm_clean)
    print(f"DTM saved: {args.out}  shape={dtm_clean.shape}  geo={geo}")

if __name__=="__main__":
    main()
'''

script_path = Path(CFG["dtm_dir"]) / "dtm_inference.py"
with open(script_path, "w") as f:
    f.write(INFER_SCRIPT)
print(f"✅ Inference script saved → {script_path}")


In [ ]:
# ─── Cell 12  Package All Outputs ────────────────────────────────────────────
from pathlib import Path
import zipfile, json

OUTPUT_FILES = [
    # Model weights (main artifact for web integration)
    (CFG["model_path"],     "model/best_model.pth"),
    # Inference config (threshold, feature params, arch)
    (CFG["inference_cfg"],  "model/inference_config.json"),
    # Inference script
    (str(Path(CFG["dtm_dir"])/"dtm_inference.py"), "model/dtm_inference.py"),
    # Training artifacts
    (CFG["history_path"],   "training/history.json"),
    (CFG["curves_path"],    "training/training_curves.png"),
    (CFG["threshold_path"], "training/threshold.json"),
    (CFG["logs_dir"]+"/threshold_curve.png",    "training/threshold_curve.png"),
    (CFG["logs_dir"]+"/sample_dtm_tiles.png",   "training/sample_dtm_tiles.png"),
]

# Also pack the first 10 sample DTM tiles
dtm_tiles = sorted(Path(CFG["dtm_dir"]).glob("val/*_dtm.npy"))[:10]
for t in dtm_tiles:
    OUTPUT_FILES.append((str(t), f"dtm_samples/{t.name}"))

out_zip = Path("/kaggle/working/dtm_model_outputs.zip")
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for src, arc in OUTPUT_FILES:
        if Path(src).exists():
            z.write(src, arcname=arc)
        else:
            print(f"  ⚠️  Missing: {src}")

print("✅ Output zip created:")
print(f"   {out_zip}  ({out_zip.stat().st_size/1e6:.1f} MB)")
print()
print("Contents:")
for src, arc in OUTPUT_FILES:
    s = "✅" if Path(src).exists() else "❌"
    print(f"  {s} {arc}")
print(f"  + {len(dtm_tiles)} DTM tile samples")
print()
print("=" * 60)
print("  WEB INTERFACE INTEGRATION")
print("=" * 60)
print()
print("  1. Download dtm_model_outputs.zip")
print("  2. model/best_model.pth   — load with DTMNet in PyTorch")
print("  3. model/inference_config.json — threshold, resolution, arch params")
print("  4. model/dtm_inference.py — standalone inference script")
print()
print("  Inference pipeline per tile:")
print("    xyz = load_lidar_tile()")
print("    feat = compute_features(xyz)          # Cell 5 logic")
print("    probs = DTMNet(feat)                  # model forward")
print("    ground_pts = xyz[probs >= threshold]  # segmentation")
print("    dtm = rasterize → fill_holes → fill_pits → smooth")
print("    → clean DTM array ready for web display / download")
print("=" * 60)
